In [ ]:
import icechunk
import matplotlib.pyplot as plt
import xarray as xr
from qa_flags import (
    flag_global_exceedances,
    flag_rsds_above_max,
    get_data,
    parse_tag,
    plot_flags,
    run_flag_loop,
    write_individual_flags,
)

from srm import catalog
from srm.cache import ArtifactCache
from srm.config import _icechunk_storage_for_path

# A. Define what data arrays exist to traverse

In [ ]:
# --- Run parameters --------------------------------------------------------
# This regional South-Africa-box run covers three GCMs, each written to its own icechunk store
# (same bucket/branch, named by GCM the same way srm.cache.ArtifactCache names pipeline output
# stores). Looping over GCMS -- rather than hardcoding one, as this notebook used to -- is what
# lets every leaf be compared against ITS OWN GCM's catalog and lineage instead of silently
# reusing whichever GCM happened to be hardcoded.


VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds", "hurs"]

In [ ]:
GCMS = ("CESM2-WACCM", "MIROC-ES2H", "UKESM")

BRANCH = "full-regional-run-issue-534"
ROOT_DIR = "s3://carbonplan-scratch/srm/output/qa/"
STORE_SUBSET_BOUNDS = (-38.0, -19.0, 13.0, 36.0)
STORE_SUBSET_ID = ArtifactCache._get_subset_id(STORE_SUBSET_BOUNDS)

# BRANCH = "v0.13.0"
# ROOT_DIR="s3://us-west-2.opendata.source.coop/carbonplan/srm-downscaling/output/production/"
# STORE_SUBSET_ID="global"

In [ ]:
def store_uri(gcm: str, root_dir: str = ROOT_DIR) -> str:
    return f"{root_dir}{gcm}-ERA5-{STORE_SUBSET_ID}.icechunk"

In [ ]:
trees: dict[str, xr.DataTree] = {}
open_errors: dict[str, str] = {}

for gcm in GCMS:
    try:
        repo = icechunk.Repository.open(_icechunk_storage_for_path(store_uri(gcm)))
        session = repo.readonly_session(BRANCH) if BRANCH else repo.readonly_session()
        trees[gcm] = xr.open_datatree(session.store, engine="zarr", chunks={})
    except Exception as exc:  # noqa: BLE001  # report every failure, do not stop at the first
        open_errors[gcm] = f"{type(exc).__name__}: {exc}"

for gcm, err in open_errors.items():
    print(f"FAILED to open {gcm}: {err}")

# GCMs whose store actually opened. A GCM whose run is still in flight is skipped here rather
# than failing the whole notebook, so this check can be run against a partially-landed run.
OPEN_GCMS = tuple(gcm for gcm in GCMS if gcm in trees)
print(f"opened {len(OPEN_GCMS)}/{len(GCMS)} stores on branch {BRANCH!r}: {', '.join(OPEN_GCMS)}")

trees[OPEN_GCMS[0]]

In [ ]:
gcms, scenarios, variables, ensembles = [], [], [], []

for gcm in OPEN_GCMS:
    tree = trees[gcm]
    for scenario_name, scenario_node in tree.children.items():
        for var_name, var_node in scenario_node.children.items():
            for ens_name, ens_node in var_node.children.items():
                gcms.append(gcm)
                scenarios.append(scenario_name)
                variables.append(var_name)
                ensembles.append(ens_name)

print(f"{len(gcms)} leaves across {len(OPEN_GCMS)} GCMs")

In [ ]:
keep_idx = [i for i, s in enumerate(scenarios) if s != "debiased_coarse"]
gcms = [gcms[i] for i in keep_idx]
scenarios = [scenarios[i] for i in keep_idx]
variables = [variables[i] for i in keep_idx]
ensembles = [ensembles[i] for i in keep_idx]

keep_idx = [i for i, v in enumerate(variables) if v != "dtr"]
gcms = [gcms[i] for i in keep_idx]
scenarios = [scenarios[i] for i in keep_idx]
variables = [variables[i] for i in keep_idx]
ensembles = [ensembles[i] for i in keep_idx]

keep_idx = [i for i, v in enumerate(variables) if v != "hurs"]
gcms = [gcms[i] for i in keep_idx]
scenarios = [scenarios[i] for i in keep_idx]
variables = [variables[i] for i in keep_idx]
ensembles = [ensembles[i] for i in keep_idx]

In [ ]:
tags = []
for i, gcm in enumerate(gcms):
    var = variables[i]
    scenario = scenarios[i]
    ens = ensembles[i]
    tag = f"{gcm}_{var}_{scenario}_{ens}"
    tags.append(tag)

# B. Generate individual QA flags

In [ ]:
PLOT_FLAG_MAPS = True
FLAG_OUTPUT_DIR = "s3://carbonplan-scratch/srm/qaqc/flags/" + STORE_SUBSET_ID + "/"

## 1. Outliers based on observations

### 1a. Calculate outlier bounds

In [ ]:
def calculate_thresholds(obs_max, obs_min, obs_max_std, obs_min_std):
    outlier_thresh_high = obs_max + (5 * obs_max_std)
    outlier_thresh_low = obs_min - (5 * obs_min_std)

    return outlier_thresh_low, outlier_thresh_high

In [ ]:
store = "s3://carbonplan-scratch/srm/qaqc/doy_obs_thresholds_global.zarr"
combined = xr.open_zarr(store)

In [ ]:
def _split(ds, suffix):
    names = [v for v in ds.data_vars if v.endswith(suffix)]
    return ds[names].rename({v: v[: -len(suffix)] for v in names})


obs_max = _split(combined, "_max")
obs_min = _split(combined, "_min")
obs_max_std = _split(combined, "_max_std")
obs_min_std = _split(combined, "_min_std")

In [ ]:
[outlier_thresh_low, outlier_thresh_high] = calculate_thresholds(
    obs_max, obs_min, obs_max_std, obs_min_std
)

### 1b. Traverse dataset and save flags

In [ ]:
from qa_flags import flag_outliers

In [ ]:
outlier_thresh_low_annual = outlier_thresh_low.min(dim="dayofyear")
outlier_thresh_high_annual = outlier_thresh_high.max(dim="dayofyear")

In [ ]:
run_flag_loop(
    tags=tags,
    trees=trees,
    flag_name="annual_outlier_flag",
    compute_flag=lambda da, var: flag_outliers(
        da=da,
        outlier_thresh_low=outlier_thresh_low_annual[var],
        outlier_thresh_high=outlier_thresh_high_annual[var],
        timescale="annual",
    ),
    flag_dir=FLAG_OUTPUT_DIR,
    write_mode="w",
)

## 2. rsds-specific latitude check

In [ ]:
CACHE_PATH = "s3://carbonplan-scratch/srm/qaqc/zonal_doy_max_rsds.zarr"


def load_cached(key: str) -> xr.DataArray | None:
    """None on a cache miss; the caller decides what to do about it."""
    try:
        return xr.open_zarr(CACHE_PATH, group=key)["data"].load()
    except FileNotFoundError:
        return None


def save_cached(key: str, da: xr.DataArray) -> None:
    da.rename("data").to_dataset().to_zarr(CACHE_PATH, group=key, mode="w")

In [ ]:
zonal_doy_max_rsds = load_cached(key="zonal_doy_max_rsds")

In [ ]:
run_flag_loop(
    tags=tags,
    trees=trees,
    flag_name="rsds_max_exceeded",
    compute_flag=lambda da, var: flag_rsds_above_max(da=da, zonal_doy_max_rsds=zonal_doy_max_rsds),
    var_filter="rsds",
    flag_dir=FLAG_OUTPUT_DIR,
)

## 3. global exceedances

In [ ]:
run_flag_loop(
    tags=tags,
    trees=trees,
    flag_name="outside_global_plausible_range",
    compute_flag=lambda da, var: flag_global_exceedances(da=da, var=var),
    flag_dir=FLAG_OUTPUT_DIR,
)

## 4. temperature inconsistencies

In [ ]:
def flag_tasmax_tas_inconsistency(tas, tasmax):
    inconsistent_days = tasmax < tas
    return inconsistent_days


def flag_tasmin_tas_inconsistency(tas, tasmin):
    inconsistent_days = tasmin > tas
    return inconsistent_days

In [ ]:
# This could be modified to fit within the run_flag_loop structure

var_filter = "tas"
plot = True

print(len(tags))
for tag in tags:
    gcm, var, scenario, ens = parse_tag(tag)
    if var_filter is not None and var != var_filter:
        continue

    tag_tasmin = f"{gcm}_tasmin_{scenario}_{ens}"
    tag_tasmax = f"{gcm}_tasmax_{scenario}_{ens}"
    if (tag_tasmin in tags) and (tag_tasmax in tags):
        print(tag)

        tas = get_data(tag=tag, trees=trees)
        tasmin = get_data(tag=tag_tasmin, trees=trees)
        tasmax = get_data(tag=tag_tasmax, trees=trees)

        flag_tas_tasmax = flag_tasmax_tas_inconsistency(tas, tasmax)
        flag_tas_tasmin = flag_tasmin_tas_inconsistency(tas, tasmin)

        flag_tas = (flag_tas_tasmax + flag_tas_tasmin) > 0
        flag_tasmin = flag_tas_tasmin
        flag_tasmax = flag_tas_tasmax

        write_individual_flags(
            flag_data=flag_tas,
            flag_name="temperature_inconsistency",
            gcm=gcm,
            var="tas",
            scenario=scenario,
            ens=ens,
            write_mode="a",
            flag_dir=FLAG_OUTPUT_DIR,
        )

        write_individual_flags(
            flag_data=flag_tasmax,
            flag_name="temperature_inconsistency",
            gcm=gcm,
            var="tasmax",
            scenario=scenario,
            ens=ens,
            write_mode="a",
            flag_dir=FLAG_OUTPUT_DIR,
        )

        write_individual_flags(
            flag_data=flag_tasmin,
            flag_name="temperature_inconsistency",
            gcm=gcm,
            var="tasmin",
            scenario=scenario,
            ens=ens,
            write_mode="a",
            flag_dir=FLAG_OUTPUT_DIR,
        )

        if plot:
            plot_flags(flags=flag_tas, time_varying=True, separate_low_high=False)
            plt.show()
            plt.close()

# C. Read in flags

In [ ]:
gcm = "CESM2-WACCM"
var = "rsds"
scenario = "ssp245"
ens = "001"

In [ ]:
flag_dir = "s3://carbonplan-scratch/srm/qaqc/flags/" + STORE_SUBSET_ID + "/"

tag = f"{gcm}_{var}_{scenario}_{ens}"
flag_ds = xr.open_dataset(flag_dir + tag + ".zarr", engine="zarr", consolidated=False)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2)
flag_frac = flag_ds["outside_global_plausible_range"].mean(dim="time")
flag_frac.where(flag_frac > 0).plot(ax=axes[0, 0])
flag_frac = flag_ds["rsds_max_exceeded"].mean(dim="time")
flag_frac.where(flag_frac > 0).plot(ax=axes[0, 1])
flag_frac = flag_ds["annual_outlier_flag"].mean(dim="time")
flag_frac.where(flag_frac > 0).plot(ax=axes[1, 0])
plt.tight_layout()

In [ ]:
flag_ds["rsds_max_exceeded"].sum(dim="time").plot()
plt.plot([28.5], [-29.6], "xr")

In [ ]:
comparison_ds = trees[gcm][scenario][var][ens]
da = comparison_ds[var]

In [ ]:
da_raw = catalog.get(gcm).to_xarray()[scenario][var].sel(ensemble_member=ens)

In [ ]:
ilat = -29.6
ilon = 28.5
pt_flagged = flag_ds["rsds_max_exceeded"].sel(lat=ilat, lon=ilon, method="nearest")
pt_da = da.sel(lat=ilat, lon=ilon, method="nearest")
pt_da_raw = da_raw.sel(lat=ilat, lon=ilon, method="nearest")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(da_raw["time.dayofyear"], pt_da_raw, ".")
plt.plot(da["time.dayofyear"], pt_da.where(pt_flagged), "o")
plt.plot(da["time.dayofyear"], pt_da, ".")

plt.subplot(1, 2, 2)
plt.plot(pt_da_raw[0 : len(pt_da)], pt_da.where(pt_flagged), "o")
plt.plot(pt_da_raw[0 : len(pt_da)], pt_da, ".")
plt.tight_layout()